In [1]:
# !pip install -U pageindex openai python-dotenv

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY= os.getenv("GROQ_API_KEY")

In [3]:
from pageindex import PageIndexClient
#from openai import OpenAI 
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

pi_client= PageIndexClient(api_key= PAGEINDEX_API_KEY)
#openai_client= OpenAI(api_key= OPENAI_API_KEY)
groqai_client = Groq(api_key=GROQ_API_KEY)

print(" PageIndex client ready")
print(" OPENAI client ready")

 PageIndex client ready
 OPENAI client ready


/Users/nikita/Documents/Vectorless_RAG/.venv/lib/python3.14/site-packages/pageindex/__init__.py:9: UserWarning: You're on Python 3.14, so an older pageindex (0.2.8.1) was installed. pageindex 0.2.10+ requires Python 3.10+.
  warnings.warn(_OLDER_INSTALL + " pageindex 0.2.10+ requires Python 3.10+.")


In [4]:
PDF_PATH= "data/sample_document.pdf"
result= pi_client.submit_document(PDF_PATH)
doc_id= result["doc_id"]
print(f"Document Id:{doc_id}")

Document Id:pi-cmv38bg5k000y0bp9de7g7641


In [5]:
# - Poll until processing is complete ----------------------------
# PageIndex builds the tree aynchronosuly. 
# For a 50- Page PDF this typically takes 30-90 seconds
import time

print("Building tree index ...")
print(" (This runs once per doucment- the index is cached for reuse)")


while True:
    status_result = pi_client.get_document(doc_id)
    status= status_result.get("status")
    print(f" Status:{status}")

    if status=="completed": 
        print("\n Tree index ready !")
        break
    elif status == "failed": 
        print("\n Processing failed, check your pdf format")
        break
    time.sleep(5)

Building tree index ...
 (This runs once per doucment- the index is cached for reuse)
 Status:processing
 Status:processing
 Status:processing
 Status:completed

 Tree index ready !


## 🔍 Section 3: Inspect the Tree Structure
### What the tree looks like:

### Document
### ├── Introduction (pages 1-3)
### │   └── Background (pages 1-2)
### ├── Financial Stability (pages 21-31)
### │   ├── Monitoring Vulnerabilities (pages 22-28)
### │   └── International Cooperation (pages 28-31)
### └── Conclusion (pages 45-47)
### Each node has:

### node_id — unique ID used during retrieval
### title — section heading
### page_index — page number in original PDF
### text — section summary (when node_summary=True)
### nodes — child sections (nested)
### This structure is what the LLM reasons over during retrieval.


In [6]:
# -- Fetch the full tree -----------------------
import json

tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree= tree_result.get("result",[])

print(f" Top-level sections:{len(pageindex_tree)}")
print("\n Raw tree (first node): ")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

 Top-level sections:1

 Raw tree (first node): 
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "start_index": 1,
  "end_index": 15,
  "summary": "This section introduces the foundational research paper for the Transformer, detailing its authors, affiliations, and the paper's introduction and abstract. It covers the background and limitations of traditional recurrent and convolutional models, presents the Transformer's novel encoder-decoder architecture based entirely on self-attention mechanisms, and analyzes the computational efficiency, parallelization, and path length advantages of self-attention. Furthermore, it outlines the training regime, experimental results and generalization capabilities in machine translation and constituency parsing, visualizations of attention heads, conclusions with future research directions, and the academic references.",
  "text": "",
  "nodes": [
    {
      "title": "Attention Is All You Need (intro)",
      "node

In [7]:
#- Pretty- print the full tree ------------------------------

def print_tree(nodes, indent=0):
    """ Recursively print tree titles for a visual overview.    """
    for node in nodes:
        prefix= " " * indent +(" | " if indent >0 else "")
        page= node.get("page_index","?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"],indent+1)

print("Full Document Structure: \n")
print_tree(pageindex_tree)

Full Document Structure: 

[0000] Attention Is All You Need (p.1)
  | [0001] Attention Is All You Need (intro) (p.1)
  | [0002] Abstract (p.1)
  | [0003] 1 Introduction (p.2)
  | [0004] 2 Background (p.2)
  | [0005] 3 Model Architecture (p.2)
   | [0006] 3 Model Architecture (intro) (p.2)
   | [0007] 3.1 Encoder and Decoder Stacks (p.3)
   | [0008] 3.2 Attention (p.3)
    | [0009] 3.2 Attention (intro) (p.3)
    | [0010] 3.2.1 Scaled Dot-Product Attention (p.4)
    | [0011] 3.2.2 Multi-Head Attention (p.4)
    | [0012] 3.2.3 Applications of Attention in our Model (p.5)
   | [0013] 3.3 Position-wise Feed-Forward Networks (p.5)
   | [0014] 3.4 Embeddings and Softmax (p.5)
   | [0015] 3.5 Positional Encoding (p.6)
  | [0016] 4 Why Self-Attention (p.6)
  | [0017] 5 Training (p.7)
  | [0018] 6 Results (p.8)
   | [0019] 6.1 Machine Translation (p.8)
   | [0020] 6.2 Model Variations (p.8)
   | [0021] 6.3 English Constituency Parsing (p.9)
  | [0022] 7 Conclusion (p.10)
  | [0023] References (

In [8]:
# ------------- Count total nodes------------------------

def count_nodes(nodes):
    total=len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print(" Each node= one retrievable section of the documenet")

Total nodes in tree: 25
 Each node= one retrievable section of the documenet


## Section 4: LLM Tree Search — The Core of PageIndex


#### This is where PageIndex fundamentally differs from vector RAG.

## Vector RAG retrieval:
### query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks


## Problem: finds what's similar, not what's relevant

### PageIndex retrieval:
##### query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
##### Advantage: LLM understands document structure, context, and intent

##### The LLM acts like a human expert scanning a Table of Contents.

In [9]:
# _______LLM Tree search function__________________


def llm_tree_search(query: str, tree: list, model: str = "qwen/qwen3.8-27b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groqai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)


In [10]:
# ------------------- Test with a sample query-------------------------------------------

query = "what is attention mechanism"

print(f" Query: {query}\n")
result= llm_tree_search(query, pageindex_tree)

print(" LLM Reasoning: ")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list",[]))

 Query: what is attention mechanism

 LLM Reasoning: 
The query asks for an explanation of the 'attention mechanism'. Looking at the document tree, Section 3.2 is titled 'Attention' (node_id 0008), which is the primary location for this topic. Specifically, the subsection '3.2 Attention (intro)' (node_id 0009) provides the general definition and description of the attention function as mapping a query and set of key-value pairs to an output. This directly answers 'what is attention mechanism'. While 'Scaled Dot-Product Attention' (0010) and 'Multi-Head Attention' (0011) provide specific implementations, the general definition is in 0009. The section on 'Why Self-Attention' (0016) explains the motivation but not the mechanism itself. Therefore, node 0009 is the most relevant, and potentially 0008 as the container.

Selected Node IDs: ['0009', '0008']


In [11]:
# --------- Helper : find nodes by ID------------------------------

def find_nodes_by_ids(tree :list, target_ids: list)-> list: 
    """ Recursively walk the tree and collect nodes matching target_ids. """

    found= []

    for node in tree: 
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node['nodes'], target_ids))
    return found

In [12]:
def generate_answer(query: str, nodes: list, model: str = "qwen/qwen3.8-27b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groqai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [13]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")

In [14]:
#- Run the full pipeline-------------------

answer = vectorless_rag( 
    query= "what is attention mechanism",
    tree= pageindex_tree
)

🔍 Query: what is attention mechanism

🧠 Reasoning: The query asks for a definition or explanation of the 'attention mechanism'. Looking at the document tree, Section 3 is 'Model Architecture' and specifically Section 3.2 is titled 'Attention'. Node 00...
🎯 Retrieved node IDs: ['0009', '0010', '0011']
📄 Sections found: ['3.2 Attention (intro)', '3.2.1 Scaled Dot-Product Attention', '3.2.2 Multi-Head Attention']

📝 Answer:
An attention function maps a query and a set of key-value pairs to a vector output, which is computed as a weighted sum of the values where the weights are determined by a compatibility function between the query and the corresponding keys (3.2 Attention (intro), Page 3).

Specifically, the **Scaled Dot-Product Attention** mechanism computes these weights by taking the dot product of the query with all keys, scaling the result by $\sqrt{d_k}$, and applying a softmax function (3.2.1 Scaled Dot-Product Attention, Page 4). This scaling prevents large dot product magnitude

In [15]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "what is Attention Is All You Need?",
    "what is the attention mechanism?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: what is Attention Is All You Need?


TypeError: 'NoneType' object is not subscriptable